In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
train=pd.read_csv(r"/content/test.csv")

In [3]:
!pip install paddleocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.8/297.8 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.6/969.6 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.0 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=663d777972364242f6191d366754d166288626e6803ac1aa5c7fac898f266361
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire


###Installing PaddlePaddle¶
To enable GPU support for PaddleOCR, we first need to install PaddlePaddle, the deep learning framework that powers PaddleOCR.

In [4]:
!pip install paddlepaddle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


# Importing Libraries and Dependencies

In this section, we import the necessary libraries for our OCR task, data handling, and image processing.

In [5]:
from paddleocr import PaddleOCR,draw_ocr
from PIL import Image
from datetime import datetime
from tqdm import tqdm
from multiprocessing import Pool
from functools import partial
import time
import urllib.request

In [6]:
ocr=PaddleOCR(use_angle_cls=True,lang='en',use_gpu=True)

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 3910/3910 [00:15<00:00, 250.80it/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10000/10000 [00:17<00:00, 571.35it/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:15<00:00, 137.74it/s]

[2025/04/14 12:40:13] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_l

# Extracting Text from Images using PaddleOCR
We define a function that takes the path to an image and uses PaddleOCR to extract the text from it.

In [7]:
## Function to extract text from images using Paddle OCR

def extract_text_from_image(image_path):
  act_text=""
  try:
    # Extract text using paddle OCR
    result=ocr.ocr(image_path,cls=True)
    # check if result is none
    if result is None:
      return "No text found or image is not valid"
    for line in result[0]:
      act_text+=str(line[1][0])+" "

  except Exception as e:
    return f"Error: {e}"
  return act_text.strip()

# 🖌️ Creating a Placeholder Image¶
This function generates a simple black placeholder image and saves it to the specified path.

In [9]:
def create_placeholder_image(image_save_path):
    try:
        placeholder_image = Image.new('RGB', (100, 100), color='black')
        placeholder_image.save(image_save_path)
    except Exception as e:
        print(f"Error creating placeholder image: {e}")

# Downloading Images with Retry Logic and Multiprocessing¶
In this section, we implement a robust image downloading mechanism:

**Image Downloading**: For each image URL, the program attempts to download the image to a specified folder. If the image already exists, the download is skipped.

**Retry Logic**: If a download fails, the process retries multiple times with a short delay between attempts. If all attempts fail, a black placeholder image is created to maintain workflow consistency.

**Multiprocessing Support**: To enhance efficiency, the code supports downloading images in parallel using multiprocessing. This feature is optional but can significantly reduce the total download time when handling large batches of images.

In [10]:
def download_image(image_link, save_folder, retries=3, delay=3):
    if not isinstance(image_link, str):
        print(f"Invalid link: {image_link}")
        return

    filename = os.path.basename(image_link)
    image_save_path = os.path.join(save_folder, filename)

    if os.path.exists(image_save_path):
        return

    for attempt in range(retries):
        try:
            urllib.request.urlretrieve(image_link, image_save_path)
            return
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {image_link}: {e}")
            time.sleep(delay)

    create_placeholder_image(image_save_path)  # Create a black placeholder image for invalid links/images

def download_images(image_links, download_folder, allow_multiprocessing=False, num_workers=10):
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)

    if allow_multiprocessing:
        download_image_partial = partial(download_image, save_folder=download_folder)

        with Pool(processes=num_workers) as pool:
            list(tqdm(pool.imap_unordered(download_image_partial, image_links), total=len(image_links)))
            pool.close()
            pool.join()
    else:
        for image_link in tqdm(image_links, total=len(image_links)):
            download_image(image_link, save_folder=download_folder)

In [12]:
import os

# 🗂️ Batch Processing and Cleanup for Efficient OCR


**Folder Cleanup**: A function to clean up a folder by removing all files after processing. This keeps the disk space usage in check and ensures that no unnecessary files linger between batches.

**Image Processing and CSV Generation:**

The images in the download folder are processed one by one, where OCR is applied to extract the text.

The results are then stored in a list, which is converted into a DataFrame and saved as a CSV file.

**Batch Download and Processing:**

The image links are split into batches (e.g., 5,000 at a time).
Each batch is downloaded, processed, and its OCR results are saved in a uniquely named CSV file.
After each batch is completed, the folder is cleaned up to prepare for the next batch, making the process memory efficient and organized.

**Batch Naming Convention**: Each CSV file generated includes the batch number and a timestamp, allowing easy identification and management of the output.

In [13]:
# Function to clean up the folder (remove processed files)
def clean_up_folder(folder):
    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)
        if os.path.isfile(file_path):
            os.remove(file_path)

# Function to process images and save text to CSV
def process_images(download_folder, output_csv):
    data = []
    for file_name in tqdm(os.listdir(download_folder), desc="Processing images"):
        image_path = os.path.join(download_folder, file_name)
        text = extract_text_from_image(image_path)
        data.append({'filename': file_name, 'text': text.strip()})

    # Create a DataFrame and save to CSV
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)

# Function to process images in batches, download, extract text, and save CSV
def download_images_in_batches(image_links, download_folder, batch_size=100, output_csv_prefix='ocr_text'):
    total_images = len(image_links)

    for start in range(0, total_images, batch_size):
        end = min(start + batch_size, total_images)
        batch_links = image_links[start:end]

        # Download images (using your download logic)
        download_images(batch_links, download_folder, allow_multiprocessing=False)

        # Generate unique CSV file name
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        batch_csv = f"{output_csv_prefix}_{start // batch_size + 1}_{timestamp}.csv"

        # Process the batch
        process_images(download_folder, batch_csv)

        # Clean up the folder but keep the folder itself
        clean_up_folder(download_folder)

        print(f"Processed batch {start // batch_size + 1} and saved to {batch_csv}")

# Ensure that the folder path is clean before starting
if os.path.exists('./train_images'):
    clean_up_folder('./train_images')

# 🚀 Executing Batch Image Processing¶
In this step, we initiate the batch image processing function to handle downloading and OCR extraction for a subset of images:

**Image Link Subset**: The function works on a specific subset of image links from index 100 to 200 in the dataset. This allows for focused processing on a manageable set of data.

**Batch Size**: The images are processed in batches of 100. This batch size strikes a balance between performance and memory efficiency, making sure the system can handle it smoothly.

**Output CSV Prefix**: The output CSV files containing the extracted text are saved with a prefix of ocr_text. Each file will also contain a unique identifier based on the batch number and timestamp, ensuring that no files are overwritten, and results remain organized.

In [14]:
download_images_in_batches(train['image_link'][100:200], './train_images/', batch_size=100, output_csv_prefix='ocr_text')

Processing images:   0%|          | 0/75 [00:00<?, ?it/s]

[2025/04/14 12:51:37] ppocr DEBUG: dt_boxes num : 4, elapsed : 0.10157036781311035
[2025/04/14 12:51:38] ppocr DEBUG: cls num  : 4, elapsed : 0.038042306900024414
[2025/04/14 12:51:38] ppocr DEBUG: rec_res num  : 4, elapsed : 0.3010382652282715


Processing images:   1%|▏         | 1/75 [00:00<00:35,  2.07it/s]

[2025/04/14 12:51:38] ppocr DEBUG: dt_boxes num : 2, elapsed : 0.49111127853393555
[2025/04/14 12:51:38] ppocr DEBUG: cls num  : 2, elapsed : 0.026740550994873047
[2025/04/14 12:51:38] ppocr DEBUG: rec_res num  : 2, elapsed : 0.10993289947509766


Processing images:   3%|▎         | 2/75 [00:01<00:42,  1.72it/s]

...

[2025/04/14 12:52:17] ppocr DEBUG: dt_boxes num : 6, elapsed : 0.12641286849975586
[2025/04/14 12:52:17] ppocr DEBUG: cls num  : 6, elapsed : 0.015168428421020508
[2025/04/14 12:52:18] ppocr DEBUG: rec_res num  : 6, elapsed : 0.3959352970123291


Processing images:  99%|█████████▊| 74/75 [00:40<00:00,  1.88it/s]

[2025/04/14 12:52:18] ppocr DEBUG: dt_boxes num : 4, elapsed : 0.13237619400024414
[2025/04/14 12:52:18] ppocr DEBUG: cls num  : 4, elapsed : 0.013417243957519531
[2025/04/14 12:52:18] ppocr DEBUG: rec_res num  : 4, elapsed : 0.3167562484741211


Processing images: 100%|██████████| 75/75 [00:40<00:00,  1.84it/s]

Processed batch 1 and saved to ocr_text_1_20250414_125137.csv


# 📝 Conclusion¶
Through this pipeline, we’ve successfully implemented a robust system for downloading, processing, and extracting text from images using PaddleOCR with GPU acceleration. By leveraging batch processing and retry logic, we ensure both efficiency and reliability. The system is designed to handle large datasets seamlessly by processing images in manageable batches and saving the OCR results to CSV files with unique identifiers.

**Key highlights include:**

**Optimized performance** through multiprocessing and GPU support for PaddleOCR.

**Error handling** and fallback mechanisms, such as placeholder images and retry attempts, ensuring minimal disruption.

**Organized workflow**, with folder cleanup after each batch to manage disk space and improve processing time.

This approach provides a scalable and efficient way to extract text from large image datasets, making it suitable for various real-world applications like document digitization, data extraction, and more.